# Train Đợt 2 - YOLO11 Fruits + Vegetables Quality (96 class)
Train tiếp từ checkpoint `best.pt` của đợt 1, gộp thêm 20 loại rau củ × 4 mức chất lượng.

**Chạy trên Google Colab** (Runtime → Change runtime type → **GPU T4**).

Thứ tự: cài đặt → kiểm tra GPU → tải dataset gộp từ Roboflow → train tiếp từ best.pt → đánh giá → test → export.

## 1. Cài đặt & kiểm tra GPU

In [ ]:
!pip install -q ultralytics roboflow
import ultralytics, torch
ultralytics.checks()
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## 2. Upload checkpoint đợt 1 (`best_dot1.pt`)
Chạy cell rồi chọn file `best_dot1.pt`. Hoặc bỏ qua nếu đã để sẵn trong Google Drive (xem cell mount bên dưới).

In [ ]:
from google.colab import files
up = files.upload()   # chọn best_dot1.pt
CKPT = list(up.keys())[0]
print('Checkpoint:', CKPT)

In [ ]:
# (TUỲ CHỌN) Dùng Google Drive thay vì upload tay
# from google.colab import drive
# drive.mount('/content/drive')
# CKPT = '/content/drive/MyDrive/NongNghiepAI/best_dot1.pt'

## 3. Tải dataset gộp từ Roboflow
**Khuyến nghị:** gộp cả 16 class cũ + 80 class mới vào CÙNG 1 project Roboflow rồi export `YOLOv11`.
Thay `API_KEY`, `workspace`, `project`, `version` bên dưới (lấy ở nút *Export → show download code*).

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key='API_KEY_CUA_BAN')
project = rf.workspace('WORKSPACE').project('PROJECT_SLUG')
version = project.version(1)
dataset = version.download('yolov11')
DATA_YAML = dataset.location + '/data.yaml'
print('data.yaml:', DATA_YAML)
!cat {DATA_YAML}

In [ ]:
# (THAY THẾ) Nếu dùng 2 dataset export riêng -> gộp bằng merge_datasets.py
# !python merge_datasets.py --d1 ./fruits_dot1 --d2 ./veg_dot2 --out ./fruits-veg-quality
# DATA_YAML = './fruits-veg-quality/data.yaml'

## 4. Kiểm tra dataset trước khi train
Đảm bảo `nc = 96` và 16 class đợt 1 nằm đúng ở đầu (index 0-15).

In [ ]:
import yaml
with open(DATA_YAML) as f: d = yaml.safe_load(f)
names = d['names']
names = [names[i] for i in sorted(names)] if isinstance(names, dict) else names
print('Số class (nc):', d.get('nc'), '| len(names):', len(names))
print('16 class đầu:', names[:16])
assert len(names) == 96, 'CẢNH BÁO: số class != 96, kiểm tra lại dataset!'

## 5. Train tiếp từ best.pt
Nạp `best_dot1.pt` (giữ trọng số backbone đã học từ đợt 1), YOLO tự khởi tạo lại head cho 96 class.
Đây là **transfer learning** - hội tụ nhanh hơn so với train từ đầu.

In [ ]:
from ultralytics import YOLO
model = YOLO(CKPT)   # nạp checkpoint đợt 1

results = model.train(
    data=DATA_YAML,
    epochs=80,            # tăng so với 50 đợt 1 vì có thêm nhiều class
    imgsz=640,
    batch=16,             # giảm còn 8 nếu hết VRAM
    patience=20,          # early stopping
    optimizer='auto',
    lr0=0.01,
    cos_lr=True,
    close_mosaic=10,
    # ----- augmentation: giúp ổn định khi data mới còn ít -----
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    fliplr=0.5, mosaic=1.0, mixup=0.1,
    project='fruit_veg_quality',
    name='dot2',
    exist_ok=True,
    seed=0,
)

## 6. Đánh giá - mAP & confusion matrix

In [ ]:
metrics = model.val()
print('mAP50-95:', round(metrics.box.map, 4))
print('mAP50   :', round(metrics.box.map50, 4))

# Hiển thị các biểu đồ ultralytics tự sinh
from IPython.display import Image, display
import glob, os
run_dir = 'fruit_veg_quality/dot2'
for img in ['confusion_matrix_normalized.png', 'results.png', 'PR_curve.png']:
    p = os.path.join(run_dir, img)
    if os.path.exists(p): display(Image(p))

In [ ]:
# mAP theo từng class -> tìm class yếu cần thêm/label lại ảnh
import numpy as np
maps = metrics.box.maps   # mAP50-95 mỗi class
order = np.argsort(maps)
print('--- 15 class YẾU NHẤT (cần bổ sung dữ liệu) ---')
for i in order[:15]:
    print(f'{names[i]:<28} mAP={maps[i]:.3f}')

## 7. Test thử trên ảnh

In [ ]:
best = 'fruit_veg_quality/dot2/weights/best.pt'
m = YOLO(best)
test_imgs = glob.glob(os.path.dirname(DATA_YAML) + '/test/images/*.jpg')[:6]
res = m.predict(test_imgs, conf=0.25, save=True)
for r in glob.glob('runs/detect/predict*/*.jpg')[:6]:
    display(Image(r))

## 8. Tải về checkpoint mới (`best.pt` đợt 2)

In [ ]:
from google.colab import files
files.download('fruit_veg_quality/dot2/weights/best.pt')
# Hoặc lưu vào Drive:
# !cp fruit_veg_quality/dot2/weights/best.pt /content/drive/MyDrive/NongNghiepAI/best_dot2.pt